In [ ]:
import numpy as np
from lib.dvr2d import *
from lib.utils import *
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (8,6)
plt.rcParams["font.size"] = 14
plt.rcParams["figure.dpi"] = 300

%load_ext autoreload

In [ ]:
r = []
theta = []
v = []
with open('mgnh.txt', 'r') as f:
# with open('h63.txt', 'r') as f:
    lines = f.readlines()
    for l in lines:
        r.append(float(l[:20]))
        theta.append(float(l[20:40]))
        v.append(float(l[40:-1]))

r = np.array(r)
theta = np.array(theta)
costheta = np.cos(theta)
v = np.array(v)
print(v.shape)
# print(theta)
print(np.unique(r))
print(np.unique(theta))
print(v)
# print(np.unique(r)[130])

In [ ]:
ind_min1 = np.argmin(v)
print(ind_min1)
ind_min2 = np.argmin(v[theta < 1])
print(theta[ind_min1], theta[ind_min2])
print(v[ind_min1], v[ind_min2])
print(r[ind_min1], r[ind_min2])

In [ ]:
ind_min1 = np.argmin(v)
print(ind_min1)
ind_min2 = np.argmin(v[theta < 0.5])
print(theta[ind_min1], theta[ind_min2])
print(v[ind_min1], v[ind_min2])
print(r[ind_min1], r[ind_min2])

In [ ]:
theta_p, r_p, v_p = theta[v < 0], r[v < 0], v[v < 0]
coef = 180 / np.pi

fig = plt.figure()
ax = plt.axes(projection='3d')
ax.plot_trisurf(theta_p[r_p < 8] * coef, r_p[r_p < 8], v_p[r_p < 8], cmap='viridis')
ax.tick_params('both', labelsize=10, pad=-2)
ax.tick_params('z', labelsize=10, pad=5)
ax.set_xlabel('$\\theta$ (degrees)', labelpad=5, fontsize=10)
ax.set_ylabel('$R$ (Å)', labelpad=0, fontsize=10)
ax.set_zlabel('$V$ (cm$^{-1}$)', labelpad=8, fontsize=10)

In [ ]:
theta_p, r_p, v_p = theta[v < 0], r[v < 0], v[v < 0]
coef = 180 / np.pi
p = plt.tricontour(theta_p[r_p < 7] * coef, r_p[r_p < 7], v_p[r_p < 7], levels=np.linspace(-110, 0, 23), colors='black', linestyles='solid')
# plt.plot(theta[ind_min1] * coef, r[ind_min1], 'x', c='black')
# plt.plot(theta[ind_min2] * coef, r[ind_min2], 'x', c='black')
# plt.plot(1.8754889808102941 * coef, 3.897, '.', c='black')
labels = p.levels[:-16:]
print(labels)
plt.clabel(p, inline=1, fontsize=5, levels=labels, fmt='%d')
plt.ylim(3, 7)
plt.xlabel('$\\theta (\degree)$')
plt.ylabel('$R$ (A)')

In [ ]:
%autoreload
pot2d = interp_pot2d(r, costheta, v)

In [ ]:
%autoreload
N_R = 200
N_theta = 20
l = 1
K_max = 0
r_min = 2
r_max = 12

In [ ]:
%autoreload
Rs_DVR, Xs_K = get_DVR_Rtheta(N_R, N_theta, 0, l=l, r_min=r_min, r_max=r_max, mol_params=mgnh_params)
Rs_angs = Rs_DVR * mgnh_params['S'] * au_to_angs
N_R_lim = Rs_angs.shape[0]
print(Rs_angs)
print(Rs_angs.shape)

In [ ]:
p = plt.tricontour(theta_p[r_p < 5.5], r_p[r_p < 5.5], v_p[r_p < 5.5], levels=np.linspace(np.min(v), 0, 25), colors='black', linestyles='solid')

Rs_grid, Xs_grid = np.meshgrid(Rs_angs, Xs_K[0])
Rs_grid, Xs_grid = Rs_grid.flatten(), Xs_grid.flatten()
Vs = pot2d(Rs_grid, Xs_grid)
plt.scatter(np.arccos(Xs_grid), Rs_grid, s=4, c='r')
plt.scatter(np.arccos(Xs_grid[Vs < -50]), Rs_grid[Vs < -50], s=4, c='b')
print(Vs)

In [ ]:
%autoreload
h_dvr = get_ham_DVR(pot2d, N_R, N_theta, 0, 0, l=l, r_min=r_min, r_max=r_max, trunc=0, mol_params=mgnh_params)
eigvals, eigvecs = np.linalg.eigh(h_dvr)
print(eigvals[:5] * hartree, eigvecs.T[0])
plt.bar(range(eigvecs.shape[1]), eigvecs.T[0])

In [ ]:
h_dvr = get_ham_DVR(pot2d, N_R, N_theta, 0, 0, l=l, r_min=r_min, r_max=r_max, trunc=2, mol_params=mgnh_params)
eigvals, eigvecs = np.linalg.eigh(h_dvr)
print(eigvals[0] * hartree, eigvecs.T[0])
plt.bar(range(eigvecs.shape[1]), -eigvecs.T[0])

In [ ]:
%autoreload
N_R = 200
N_theta = 20
l = 1
K_max = 0

r_min = 2.0
r_max = 12

Rs_DVR, Xs_K = get_DVR_Rtheta(N_R, N_theta, 0, l=l, r_min=r_min, r_max=r_max, mol_params=mgnh_params)
N_R_lim = Rs_DVR.shape[0]
h_dvr = get_ham_DVR(pot2d, N_R, N_theta, 0, 0, l=l, r_min=r_min, r_max=r_max, trunc=0, mol_params=mgnh_params)
eigvals, eigvecs = np.linalg.eigh(h_dvr)
print(f'[{r_min:.1f}, {r_max:.1f}], N_R = {N_R_lim}, N_theta = {N_theta}: {eigvals[0] * hartree}')
ref_energy = eigvals[0] * hartree

In [ ]:
for i in range(10):
    print(f'E{i} - E0 = {eigvals[i] * hartree - ref_energy}')

In [ ]:
%autoreload
N_R = 200
N_theta = 10
l = 1
K_max = 0

r_min = 3.0
r_max_list = [4, 4.5, 5, 6, 7, 8, 10, 12]
for r_max in r_max_list:
    Rs_DVR, Xs_K = get_DVR_Rtheta(N_R, N_theta, 0, l=l, r_min=r_min, r_max=r_max, mol_params=mgnh_params)
    N_R_lim = Rs_DVR.shape[0]
    h_dvr = get_ham_DVR(pot2d, N_R, N_theta, 0, 0, l=l, r_min=r_min, r_max=r_max, trunc=0, mol_params=mgnh_params)
    eigvals, eigvecs = np.linalg.eigh(h_dvr)
    print(f'[{r_min:.1f}, {r_max:.1f}], N_R = {N_R_lim}, N_theta = {N_theta}: {eigvals[0] * hartree:.5f}, diff = {eigvals[0] * hartree - ref_energy:.5f}')

In [ ]:
%autoreload
N_R = 200
N_theta = 20
l = 1
K_max = 0

r_min = 3.0
r_max_list = [4, 4.5, 5, 6, 7, 8, 10, 12]
for r_max in r_max_list:
    Rs_DVR, Xs_K = get_DVR_Rtheta(N_R, N_theta, 0, l=l, r_min=r_min, r_max=r_max, mol_params=mgnh_params)
    N_R_lim = Rs_DVR.shape[0]
    h_dvr = get_ham_DVR(pot2d, N_R, N_theta, 0, 0, l=l, r_min=r_min, r_max=r_max, trunc=0, mol_params=mgnh_params)
    eigvals, eigvecs = np.linalg.eigh(h_dvr)
    print(f'[{r_min:.1f}, {r_max:.1f}], N_R = {N_R_lim}, N_theta = {N_theta}: {eigvals[0] * hartree:.5f}, diff = {eigvals[0] * hartree - ref_energy:.5f}')

In [ ]:
%autoreload
N_R = 200
N_theta = 10
l = 1
K_max = 0

r_min_list = [2.0, 3.0, 3.2, 3.4, 3.6, 3.8, 4]
r_max = 6
for r_min in r_min_list:
    Rs_DVR, Xs_K = get_DVR_Rtheta(N_R, N_theta, 0, l=l, r_min=r_min, r_max=r_max, mol_params=mgnh_params)
    N_R_lim = Rs_DVR.shape[0]
    h_dvr = get_ham_DVR(pot2d, N_R, N_theta, 0, 0, l=l, r_min=r_min, r_max=r_max, trunc=0, mol_params=mgnh_params)
    eigvals, eigvecs = np.linalg.eigh(h_dvr)
    print(f'[{r_min:.1f}, {r_max:.1f}], N_R = {N_R_lim}, N_theta = {N_theta}: {eigvals[0] * hartree:.5f}, diff = {eigvals[0] * hartree - ref_energy:.5f}')

In [ ]:
%autoreload
N_R = 200
l = 1
K_max = 0

r_min = 3.0
r_max_list = 6
N_thetas = [20, 10, 8, 5, 4, 3, 2]
for N_theta in N_thetas:
    Rs_DVR, Xs_K = get_DVR_Rtheta(N_R, N_theta, 0, l=l, r_min=r_min, r_max=r_max, mol_params=mgnh_params)
    N_R_lim = Rs_DVR.shape[0]
    h_dvr = get_ham_DVR(pot2d, N_R, N_theta, 0, 0, l=l, r_min=r_min, r_max=r_max, trunc=0, mol_params=mgnh_params)
    eigvals, eigvecs = np.linalg.eigh(h_dvr)
    print(f'[{r_min:.1f}, {r_max:.1f}], N_R = {N_R_lim}, N_theta = {N_theta}: {eigvals[0] * hartree:.5f}, diff = {eigvals[0] * hartree - ref_energy:.5f}')

In [ ]:
%autoreload

N_R = 200
N_theta = 8
l = 1
K_max = 0

r_min = 3.0
r_max = 6
truncs = [0, 20, 10, 5, 3, 2, 1]
for t in truncs:
    Rs_DVR, Xs_K = get_DVR_Rtheta(N_R, N_theta, 0, l=l, r_min=r_min, r_max=r_max, mol_params=mgnh_params)
    N_R_lim = Rs_DVR.shape[0]
    h_dvr = get_ham_DVR(pot2d, N_R, N_theta, 0, 0, l=l, r_min=r_min, r_max=r_max, trunc=t, mol_params=mgnh_params)
    eigvals, eigvecs = np.linalg.eigh(h_dvr)
    print(f'[{r_min:.1f}, {r_max:.1f}], N_R = {N_R_lim}, P = {t if t != 0 else N_R_lim}, N_theta = {N_theta}: {eigvals[0] * hartree:.5f}, diff = {eigvals[0] * hartree - ref_energy:.5f}')

In [ ]:
%autoreload

N_R = 200
l = 1
K_max = 0

r_min = 3.0
r_max = 12
t = 0
N_thetas = [20, 10, 8, 5, 4, 3, 2]
for N_theta in N_thetas:
    Rs_DVR, Xs_K = get_DVR_Rtheta(N_R, N_theta, 0, l=l, r_min=r_min, r_max=r_max, mol_params=mgnh_params)
    N_R_lim = Rs_DVR.shape[0]
    h_dvr = get_ham_DVR(pot2d, N_R, N_theta, 0, 0, l=l, r_min=r_min, r_max=r_max, trunc=t, mol_params=mgnh_params)
    eigvals, eigvecs = np.linalg.eigh(h_dvr)
    print(f'[{r_min:.1f}, {r_max:.1f}], N_R = {N_R_lim}, P = {t if t != 0 else N_R_lim}, N_theta = {N_theta}: {eigvals[0] * hartree:.5f}, diff = {eigvals[0] * hartree - ref_energy:.5f}')

In [ ]:
%autoreload
N_R = 200
N_theta = 10
l = 1
K_max = 0
P = 0
r_min = 3.0
r_max = 6.0

Rs_DVR, Xs_K = get_DVR_Rtheta(N_R, N_theta, 0, l=l, r_min=r_min, r_max=r_max, mol_params=mgnh_params)
N_R_lim = Rs_DVR.shape[0]
h_dvr = get_ham_DVR(pot2d, N_R, N_theta, 0, 0, l=l, r_min=r_min, r_max=r_max, trunc=P, mol_params=mgnh_params)
eigvals, eigvecs = np.linalg.eigh(h_dvr)
# print(f'[{r_min:.1f}, {r_max:.1f}], N_R = {N_R_lim}, N_theta = {N_theta}: {eigvals[0] * hartree}, diff = {eigvals[0] * hartree - ref_energy:.5f}')

In [ ]:
eigvals, eigvecs = np.linalg.eigh(h_dvr)
print(eigvals[0] * hartree, eigvecs.T[0])
plt.bar(range(eigvecs.shape[1]), eigvecs.T[0])

In [ ]:
vec = eigvecs.T[0]
vec = vec.reshape((P if P > 0 else N_R_lim, N_theta))
# vec = vec.reshape(-1)

In [ ]:
i = 2
plt.bar(range(N_R_lim), eigvecs.T[0][i * N_R_lim:(i+1) * N_R_lim])


In [ ]:
eigvecs.T[0].shape

In [ ]:
np.linspace(np.arccos(Xs_K[0])[-1], np.pi, 24 * 12).shape

In [ ]:
%autoreload
P = 5

# Rs_DVR5, Xs_K5 = get_DVR_Rtheta(N_R, N_theta, 0, l=l, r_min=r_min, r_max=r_max)
# N_R_lim5 = Rs_DVR.shape[0]
h_dvr5 = get_ham_DVR(pot2d, N_R, N_theta, 0, 0, l=l, r_min=r_min, r_max=r_max, trunc=P, mol_params=mgnh_params)
eigvals5, eigvecs5 = np.linalg.eigh(h_dvr5)
# print(f'[{r_min:.1f}, {r_max:.1f}], N_R = {N_R_lim}, N_theta = {N_theta}: {eigvals[0] * hartree}, diff = {eigvals[0] * hartree - ref_energy:.5f}')

In [ ]:
# eigvals, eigvecs = np.linalg.eigh(h_dvr)
# print(eigvals[0], eigvecs.T[0])
plt.bar(range(eigvecs5.shape[1]), eigvecs5.T[0])

In [ ]:
eigvecs5.shape

In [ ]:
cmap = plt.get_cmap('tab10')

p = plt.tricontour(theta_p[r_p < 6], r_p[r_p < 6], v_p[r_p < 6], levels=np.linspace(-110, 0, 23), colors='black', linestyles='solid', alpha=0.8)
# plt.plot(theta[ind_min1], r[ind_min1], 'x', c='black')
# plt.plot(theta[ind_min2], r[ind_min2], 'x', c='black')
# plt.plot(1.8754889808102941, 3.897, 'o', c='black')
Rs_grid, Xs_grid = np.meshgrid(Rs_DVR * mgnh_params['S'] * au_to_angs, np.arccos(Xs_K[0]))
Rs_grid, Xs_grid = Rs_grid.flatten(), Xs_grid.flatten()

labels = p.levels[:-15:]
plt.clabel(p, inline=1, fontsize=5, levels=labels, fmt='%.2f', use_clabeltext=True)
plt.ylim()
plt.xlabel('$\\theta$')
plt.ylabel('$R$ (A)')
plt.ylim((3, 6))
plt.xlim((0, np.pi))

offset = 0.05
plt.bar(np.arccos(Xs_K[0]), eigvecs5.T[0][::5] * 1.5, bottom=5, width=0.05, color='black', alpha=0.4)
plt.bar(np.arccos(Xs_K[0]) + 1 * offset, eigvecs5.T[0][1::5] * 1.5, bottom=5, width=0.05, color='black', alpha=0.4)
plt.bar(np.arccos(Xs_K[0]) + 2 * offset, eigvecs5.T[0][2::5] * 1.5, bottom=5, width=0.05, color='black', alpha=0.4)
plt.bar(np.arccos(Xs_K[0]) + 3 * offset, eigvecs5.T[0][3::5] * 1.5, bottom=5, width=0.05, color='black', alpha=0.4)
plt.bar(np.arccos(Xs_K[0]) + 4 * offset, eigvecs5.T[0][4::5] * 1.5, bottom=5, width=0.05, color='black', alpha=0.4)
# plt.axhline(4.0, linestyle='dashed', c='black', linewidth=0.2)

for i in range(N_theta):
    evec = eigvecs.T[0][i * N_R_lim:(i+1) * N_R_lim]
    epos = np.where(evec >=0, evec, 0)
    eneg = np.where(evec <0, -evec, 0)
    plt.scatter(np.repeat(np.arccos(Xs_K[0][i]), N_R_lim), Rs_DVR * mgnh_params['S'] * au_to_angs, marker='o', facecolors='black', edgecolors='black', s=epos * 600, zorder=20, alpha=0.7)
    plt.scatter(np.repeat(np.arccos(Xs_K[0][i]), N_R_lim), Rs_DVR * mgnh_params['S'] * au_to_angs, facecolors='none', edgecolors='black', s=eneg * 600, alpha=0.7, zorder=20)

In [ ]:
from scipy.optimize import minimize

x = 0
f = lambda r: pot2d([r], [x]) * hartree
minimize(f, 3.5)